In [ ]:
import networkx as nx
from rdflib import Graph, BNode, URIRef, Literal, RDF
from pyvis import network as net

In [ ]:
FULL_GRAPH = True

In [ ]:
g = Graph()

files = [
    "/home/japlan001/PycharmProjects/metadata_converter/output/Action_SAMEA112553379.jsonld",
    "/home/japlan001/PycharmProjects/metadata_converter/output/Product_SAMEA112553379.jsonld"
]

for file in files:
    g.parse(file, format="json-ld")

if FULL_GRAPH:
    g.parse("graph.jelly", format="jelly")

In [ ]:
RDF.type

In [ ]:
for subject, predicate, object in g.triples((None, None, None)):
    if ".jsonld" in str(object):
        print(subject, predicate, object)

In [ ]:
def short_label(node) -> str:
    """
    Return a concise, human-readable label for an RDF node.

    Strips the namespace prefix from URIs, keeping only the fragment
    (``#name``) or last path segment (``/name``).

    Parameters
    ----------
    node : rdflib.term.Node
        An RDF node — one of ``URIRef``, ``Literal``, or ``BNode``.

    Returns
    -------
    str
        Shortened string representation suitable for use as a node label.
    """
    if isinstance(node, URIRef):
        uri = str(node)
        if "#" in uri:
            return uri.split("#")[-1]
        # if something went wrong during resolution and no / is between the context url and the type
        elif "schema.org" in uri:
            return uri.split("schema.org")[-1]
        return uri.rstrip("/").split("/")[-1] or uri
    if isinstance(node, Literal):
        return str(node)
    # mark blank nodes
    if isinstance(node, BNode):
        return f"_:{str(node)}"
    return str(node)


# Map RDF types → node shapes (pyvis supports: diamond, dot, star, triangle, triangleDown, square and icon with label outside)
TYPE_SHAPES = {
    "Product": "triangle",
    "Action": "square",
    "Dataset": "star",
    "Person": "diamond"
}

In [ ]:
class FilteredGraph:
    def __init__(self, graph: Graph):
        self._graph = graph
        self._added_nodes = set()
        self.nx_graph = nx.DiGraph()

        for subject, predicate, object in self._graph:
            # skip the b5d project and the grant for the full graph
            if FULL_GRAPH and "b5d" in str(object) or "b5d" in str(subject):
                continue

            # add all major nodes
            if self.valid(subject):
                self.add_node(subject)

            if self.valid(object):
                self.fill_intermediate_nodes(subject, predicate, object)

    @staticmethod
    def valid(entity)->bool:
        if ".jsonld" in str(entity):
            return True
        return False

    def add_node(self, node_uri: URIRef | BNode)->None:
        if node_uri not in self._added_nodes:
            label = short_label(node_uri)
            node_type = short_label(g.value(node_uri, RDF.type))
            # use dot if shape not explicitly stated
            shape = TYPE_SHAPES.get(node_type, "dot")

            tooltip = f"URI: {node_uri}\nType: {node_type}"
            for s, p, o  in g.triples((node_uri, None, None)):
                if "name" in p:
                    tooltip += f"\nName: {short_label(o)}"
                if "description" in p:
                    tooltip += f"\nDescription: {short_label(o)}"

            self.nx_graph.add_node(short_label(node_uri), label=label, shape=shape, title=tooltip)
            self._added_nodes.add(node_uri)

    def fill_intermediate_nodes(self, s, p, o)->None:
        self.add_node(o)
        self.nx_graph.add_edge(short_label(s), short_label(o), label=short_label(p))
        # we found a major node and can stop now
        if self.valid(s):
            return
        # else we build the intermediate nodes
        else:
            for base_triples in self._graph.triples((None, None, s)):
                s, p, o = base_triples
                self.fill_intermediate_nodes(s, p, o)

In [ ]:
G = FilteredGraph(g).nx_graph
for n in G.nodes:
    print(n)

In [ ]:
pyg = net.Network(
    notebook=False,           # export for browser, not notebook cell
    cdn_resources="remote",   # or "in_line"
    width="90%",              # NOT 100% width or 100% height
    height="800px"            # give it finite space
)
pyg.from_nx(G)

# activate the panel, then show in browser
pyg.show_buttons()
pyg.show("Test.html", notebook=False)   # opens in browser